# 13 — Date-Level Uncertainty and Robustness Analysis

This notebook quantifies finite-sample uncertainty around the locked empirical
results. The settlement date is the uncertainty unit.

The weather model, continuous calibration, probability calibration and trading
strategy are not reselected.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        manifest_path = (
            candidate
            / "data/manifests/"
            "13_uncertainty_analysis_manifest.json"
        )

        if manifest_path.exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


ROOT = locate_repository(Path.cwd())

manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "13_uncertainty_analysis_manifest.json"
    ).read_text(encoding="utf-8")
)

summary = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "13_uncertainty_main_table.csv"
)

rule_summary = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "13_uncertainty_rule_sensitivity.csv"
)

date_effects = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "13_uncertainty_date_effect_panel.csv"
)

source_resolution = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "13_uncertainty_source_resolution.csv"
)

integrity = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "13_uncertainty_integrity_checks.csv"
)

trading = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "12_locked_trading_date_panel.csv"
)

print("Status:", manifest["status"])
print(
    "Uncertainty unit:",
    manifest["uncertainty_unit"],
)
print(
    "Bootstrap replicates:",
    manifest["bootstrap_replicates"],
)
print(
    "Estimands:",
    manifest["estimand_count"],
)

Status: DATE_LEVEL_UNCERTAINTY_ANALYSIS_COMPLETE
Uncertainty unit: settlement_date
Bootstrap replicates: 20000
Estimands: 6


## Effect definitions

For loss scores, the reported date-level effect is

\[
D_d
=
L^{\mathrm{baseline}}_d
-
L^{\mathrm{comparator}}_d.
\]

A positive value therefore favours the locked weather model or the regularised
probability vector.

For trading, the effect is the realised net payoff relative to the no-trade
benchmark:

\[
D_d
=
\Pi^{\mathrm{net}}_d.
\]

All quantities are first reduced to one observation per settlement date.

In [2]:
print(
    source_resolution.to_string(
        index=False
    )
)

      source_name                                                               path  rows date_column     block_column     block_source   rule_column  distinct_dates                       blocks                                           decision_rules            resolved_metric_1                 resolved_metric_2             resolved_metric_3                  resolved_metric_4
       continuous outputs/diagnostics/07_continuous_prediction_and_outcome_panel.csv   477 target_date chronology_block chronology_block decision_rule              40 ['external_test', 'holdout'] ['12h_prior', '24h_prior', '6h_prior', 'event_day_open']             raw_crps_derived            model_crps_99q_derived                           NaN                                NaN
categorical_model          outputs/diagnostics/10_locked_categorical_score_panel.csv   159 target_date chronology_block chronology_block decision_rule              40 ['external_test', 'holdout'] ['12h_prior', '24h_prior', '6h_prior', 'ev

## Main date-level uncertainty results

The interval is a fixed-seed percentile interval from a non-parametric
bootstrap over settlement dates. The sign-flip calculation is reported as a
descriptive paired randomisation diagnostic.

In [3]:
display_columns = [
    "estimand_label",
    "chronology_block",
    "settlement_dates",
    "mean_effect",
    "standard_error",
    "bootstrap_lower",
    "bootstrap_upper",
    "median_effect",
    "positive_effect_share",
    "sign_flip_two_sided_p_value",
]

print(
    summary[
        display_columns
    ].to_string(
        index=False
    )
)

                              estimand_label chronology_block  settlement_dates  mean_effect  standard_error  bootstrap_lower  bootstrap_upper  median_effect  positive_effect_share  sign_flip_two_sided_p_value
                 Raw minus locked-model CRPS          holdout                10     1.411113        0.233908         0.938603         1.794626       1.526615               0.900000                     0.003906
                 Raw minus locked-model CRPS    external_test                30     1.196003        0.148831         0.900798         1.481071       1.214856               0.900000                     0.000010
 Raw minus regularised categorical log score          holdout                10    -0.140714        0.146420        -0.388502         0.149661      -0.361607               0.300000                     0.357422
 Raw minus regularised categorical log score    external_test                30    -0.521925        0.105034        -0.745918        -0.343338      -0.380720   

## Interpretation by empirical question

The six estimands address separate questions:

1. whether the locked distribution improves continuous CRPS relative to the
   raw deterministic forecast;
2. whether uniform probability mixing changes categorical log and Brier
   scores;
3. whether the weather probabilities outperform the market on exact common
   support;
4. whether the locked trading rule produces positive reduced-form net payoff.

The effects are not pooled across these questions because their units and
interpretations differ.

In [4]:
for estimand in summary["estimand"].drop_duplicates():
    print()
    print("=" * 88)
    print(estimand)
    print("=" * 88)

    subset = summary.loc[
        summary["estimand"].eq(estimand),
        [
            "chronology_block",
            "settlement_dates",
            "mean_effect",
            "bootstrap_lower",
            "bootstrap_upper",
            "positive_effect_share",
            "sign_flip_two_sided_p_value",
        ],
    ]

    print(
        subset.to_string(
            index=False
        )
    )


continuous_crps_improvement
chronology_block  settlement_dates  mean_effect  bootstrap_lower  bootstrap_upper  positive_effect_share  sign_flip_two_sided_p_value
         holdout                10     1.411113         0.938603         1.794626                    0.9                     0.003906
   external_test                30     1.196003         0.900798         1.481071                    0.9                     0.000010

categorical_log_score_improvement
chronology_block  settlement_dates  mean_effect  bootstrap_lower  bootstrap_upper  positive_effect_share  sign_flip_two_sided_p_value
         holdout                10    -0.140714        -0.388502         0.149661               0.300000                     0.357422
   external_test                30    -0.521925        -0.745918        -0.343338               0.033333                     0.000010

categorical_brier_improvement
chronology_block  settlement_dates  mean_effect  bootstrap_lower  bootstrap_upper  positive_effect_sh

## Decision-rule sensitivity

Decision-rule results are secondary diagnostics. The principal uncertainty
unit remains the settlement date, and no decision rule is reselected from
these results.

In [5]:
if len(rule_summary):
    print(
        rule_summary.to_string(
            index=False
        )
    )
else:
    print(
        "No decision-rule sensitivity rows "
        "were available."
    )

                                estimand                               estimand_label chronology_block  decision_rule  settlement_dates  mean_effect  standard_error  median_effect  positive_effect_share  minimum_effect  maximum_effect
             continuous_crps_improvement                  Raw minus locked-model CRPS          holdout      24h_prior                10     1.259089        0.317791       1.696690               0.800000       -0.429825        2.215630
             continuous_crps_improvement                  Raw minus locked-model CRPS          holdout      12h_prior                10     1.431286        0.303716       1.652498               0.900000       -1.008613        2.354897
             continuous_crps_improvement                  Raw minus locked-model CRPS          holdout       6h_prior                10     1.451362        0.260326       1.661842               0.900000       -0.429825        2.215630
             continuous_crps_improvement                  Ra

## Locked trading paths

The trading strategy remains the development-selected strategy from Notebook
12. The following table displays the cumulative net payoff path without
changing the rule, threshold or cost assumption.

In [6]:
trading["target_date"] = pd.to_datetime(
    trading["target_date"],
    errors="coerce",
)

trading["net_payoff"] = pd.to_numeric(
    trading["net_payoff"],
    errors="coerce",
).fillna(0.0)

trading = trading.sort_values(
    [
        "chronology_block",
        "target_date",
    ]
)

trading["cumulative_net_payoff"] = (
    trading.groupby(
        "chronology_block"
    )["net_payoff"]
    .cumsum()
)

print(
    trading[
        [
            "target_date",
            "chronology_block",
            "market_support_available",
            "trade",
            "net_payoff",
            "cumulative_net_payoff",
        ]
    ].to_string(
        index=False
    )
)

target_date chronology_block  market_support_available  trade  net_payoff  cumulative_net_payoff
 2026-06-01    external_test                      True   True     -0.1050                -0.1050
 2026-06-02    external_test                      True   True     -0.1350                -0.2400
 2026-06-03    external_test                      True   True     -0.0225                -0.2625
 2026-06-04    external_test                      True   True      0.4000                 0.1375
 2026-06-05    external_test                      True   True     -0.2000                -0.0625
 2026-06-06    external_test                     False  False      0.0000                -0.0625
 2026-06-07    external_test                     False  False      0.0000                -0.0625
 2026-06-08    external_test                     False  False      0.0000                -0.0625
 2026-06-09    external_test                      True   True     -0.1250                -0.1875
 2026-06-10    external_test  

## Evidential boundary

The bootstrap intervals and sign-flip calculations describe uncertainty within
the observed dates. They do not establish population-level profitability,
market inefficiency or executable trading performance.

Order-book depth, spread crossing, partial fills, latency and market impact
remain outside the reduced-form trading exercise.

In [7]:
def as_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(
            {
                "true",
                "1",
                "yes",
                "y",
            }
        )
    )


assert as_bool(
    integrity["passed"]
).all()

assert (
    manifest["weather_model_reselected"]
    is False
)

assert (
    manifest[
        "continuous_calibration_reselected"
    ]
    is False
)

assert (
    manifest[
        "probability_calibration_reselected"
    ]
    is False
)

assert (
    manifest[
        "trading_strategy_reselected"
    ]
    is False
)

assert (
    manifest[
        "strategy_refitted_before_external_test"
    ]
    is False
)

assert (
    manifest[
        "formal_population_inference_claimed"
    ]
    is False
)

assert (
    manifest[
        "market_inefficiency_claimed"
    ]
    is False
)

assert (
    manifest[
        "executable_profitability_claimed"
    ]
    is False
)

print(
    "All Notebook 13 integrity checks passed:",
    True,
)

print(
    "Model or calibration reselection:",
    False,
)

print(
    "Trading strategy reselection:",
    False,
)

All Notebook 13 integrity checks passed: True
Model or calibration reselection: False
Trading strategy reselection: False
